In [1]:
import os
from pyspark.sql import SparkSession
from datetime import datetime
import pytz
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import pytz
from delta import configure_spark_with_delta_pip

# Lendo variáveis do ambiente do container
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
SPARK_MASTER = os.getenv("SPARK_MASTER", "spark://spark-master:7077")

# Define Delta Lake version compatible with your Spark
DELTA_VERSION = "3.2.0"

builder = (
    SparkSession.builder
    .appName("base_atraso")
    .master(SPARK_MASTER)
    
    # Add Delta Lake packages explicitly
    .config("spark.jars.packages", f"io.delta:delta-spark_2.12:{DELTA_VERSION},io.delta:delta-storage:{DELTA_VERSION}")
    
    # Delta Lake SQL extensions
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    
    # MinIO / S3
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    
    # Additional Delta configs for S3
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore")
    .config("spark.sql.parquet.compression.codec", "snappy")
    
    # Optional: Hadoop AWS configuration
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
)

# Configure with Delta pip
spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-79f37bbd-fc67-493e-943d-e5469128ca0a;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 172ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

In [2]:
agora=datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc=agora.strftime("%Y%m%d%H%M%S")

In [3]:
path = "s3a://bronze/book_atraso/"
df_book_atraso = spark.read.parquet(path)
df_book_atraso.show(5, truncate=False)

25/12/30 21:17:09 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
25/12/30 21:17:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+------------------+----------------------------------------------------------------+------------------+---------+-------------+------------------------+--------------+-------+--------+---------------------+---------+---------------------+---------------------+-------------------+------------------------+-------------------+--------------+--------------------------+----------------------------+--------------------+---------------------+----------------------+------------------+------------------+------------------+----------------------+----------------+------------------+-------------------+------+-------+--------+-------+----------------+----------+---------------+-------------+---------------+--------------+----------------+-----------------------+--------------+------------------+---------------+----------------------+---------------------+-----------------+----------------------+------------------+
|NUM_CPF    |DAT_REFERENCIA    |NUM_FATURA_HASH                        

In [4]:
df_book_atraso.createOrReplaceTempView("raw_00")

In [5]:
df_book_atraso.count()

31611316

In [6]:

raw_00_com_safra = spark.sql("""
    SELECT
        *,
        CAST(
            date_format(
                to_timestamp(DAT_CRIACAO_DW, 'ddMMMyyyy:HH:mm:ss'),
                'yyyyMM'
            ) AS INT
        ) AS SAFRA
    FROM raw_00
""")

raw_00_com_safra.createOrReplaceTempView("raw_00_com_safra")

In [7]:
raw_00_com_safra.printSchema()

root
 |-- NUM_CPF: string (nullable = true)
 |-- DAT_REFERENCIA: string (nullable = true)
 |-- NUM_FATURA_HASH: string (nullable = true)
 |-- NUM_ENT_SEQ_FATURA: string (nullable = true)
 |-- CONTRATO: string (nullable = true)
 |-- DW_UN_NEGOCIO: string (nullable = true)
 |-- DW_HIS_PONTO_VENDA_COMTA: string (nullable = true)
 |-- DW_NUM_CLIENTE: string (nullable = true)
 |-- DW_AREA: string (nullable = true)
 |-- DW_CICLO: string (nullable = true)
 |-- DW_TIPO_CLIENTE_CONTA: string (nullable = true)
 |-- DW_OFERTA: string (nullable = true)
 |-- DW_FAIXA_AGING_FATURA: string (nullable = true)
 |-- DW_FAIXA_AGING_DIVIDA: string (nullable = true)
 |-- DW_FAIXA_TEMPO_BASE: string (nullable = true)
 |-- DW_FAIXA_AGING_PROX_FECH: string (nullable = true)
 |-- DW_TIPO_FATURAMENTO: string (nullable = true)
 |-- COD_PLATAFORMA: string (nullable = true)
 |-- DAT_CRIACAO_REGISTRO_TRANS: string (nullable = true)
 |-- DAT_ALTERACAO_REGISTRO_TRANS: string (nullable = true)
 |-- DAT_CANCELAMENTO_FAT

In [8]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos,
        COUNT(DISTINCT CONTRATO) as contrato_distintos,
        COUNT(DISTINCT DW_NUM_CLIENTE) as num_tel_distintos
    FROM raw_00_com_safra
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

contagem_safras.show(5, truncate=False)

+------+------------+-------------+------------------+-----------------+
|SAFRA |total_linhas|cpf_distintos|contrato_distintos|num_tel_distintos|
+------+------------+-------------+------------------+-----------------+
|202311|1173302     |572247       |655007            |655204           |
|202312|1152922     |569431       |651600            |651739           |
|202401|2278589     |634741       |731515            |733220           |
|202402|1055098     |554201       |627401            |627387           |
|202403|1074593     |558201       |631715            |631699           |
+------+------------+-------------+------------------+-----------------+
only showing top 5 rows



In [9]:
print('lista de colunas para tipar')
for col in spark.table("raw_00_com_safra").columns:
    print('cast(' + col + ' as) as ' + col + ',')

lista de colunas para tipar
cast(NUM_CPF as) as NUM_CPF,
cast(DAT_REFERENCIA as) as DAT_REFERENCIA,
cast(NUM_FATURA_HASH as) as NUM_FATURA_HASH,
cast(NUM_ENT_SEQ_FATURA as) as NUM_ENT_SEQ_FATURA,
cast(CONTRATO as) as CONTRATO,
cast(DW_UN_NEGOCIO as) as DW_UN_NEGOCIO,
cast(DW_HIS_PONTO_VENDA_COMTA as) as DW_HIS_PONTO_VENDA_COMTA,
cast(DW_NUM_CLIENTE as) as DW_NUM_CLIENTE,
cast(DW_AREA as) as DW_AREA,
cast(DW_CICLO as) as DW_CICLO,
cast(DW_TIPO_CLIENTE_CONTA as) as DW_TIPO_CLIENTE_CONTA,
cast(DW_OFERTA as) as DW_OFERTA,
cast(DW_FAIXA_AGING_FATURA as) as DW_FAIXA_AGING_FATURA,
cast(DW_FAIXA_AGING_DIVIDA as) as DW_FAIXA_AGING_DIVIDA,
cast(DW_FAIXA_TEMPO_BASE as) as DW_FAIXA_TEMPO_BASE,
cast(DW_FAIXA_AGING_PROX_FECH as) as DW_FAIXA_AGING_PROX_FECH,
cast(DW_TIPO_FATURAMENTO as) as DW_TIPO_FATURAMENTO,
cast(COD_PLATAFORMA as) as COD_PLATAFORMA,
cast(DAT_CRIACAO_REGISTRO_TRANS as) as DAT_CRIACAO_REGISTRO_TRANS,
cast(DAT_ALTERACAO_REGISTRO_TRANS as) as DAT_ALTERACAO_REGISTRO_TRANS,
cast(DAT_CAN

In [10]:
lake = spark.sql(     
    """
        select
        
            -- campos do arquivo --

            try_cast(NUM_CPF as STRING) as NUM_CPF,
            try_cast(SAFRA as INT) as SAFRA,
            case 
                when trim(DAT_REFERENCIA) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_REFERENCIA), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_REFERENCIA,
            try_cast(NUM_FATURA_HASH as STRING) as NUM_FATURA_HASH,
            try_cast(NUM_ENT_SEQ_FATURA as INT) as NUM_ENT_SEQ_FATURA,
            try_cast(CONTRATO as BIGINT) as CONTRATO,
            try_cast(DW_UN_NEGOCIO as INT) as DW_UN_NEGOCIO,
            try_cast(DW_HIS_PONTO_VENDA_COMTA as BIGINT) as DW_HIS_PONTO_VENDA_COMTA,
            try_cast(DW_NUM_CLIENTE as BIGINT) as DW_NUM_CLIENTE,
            try_cast(DW_AREA as INT) as DW_AREA,
            try_cast(DW_CICLO as INT) as DW_CICLO,
            try_cast(DW_TIPO_CLIENTE_CONTA as INT) as DW_TIPO_CLIENTE_CONTA,
            try_cast(DW_OFERTA as INT) as DW_OFERTA,
            try_cast(DW_FAIXA_AGING_FATURA as INT) as DW_FAIXA_AGING_FATURA,
            try_cast(DW_FAIXA_AGING_DIVIDA as INT) as DW_FAIXA_AGING_DIVIDA,
            try_cast(DW_FAIXA_TEMPO_BASE as INT) as DW_FAIXA_TEMPO_BASE,
            try_cast(DW_FAIXA_AGING_PROX_FECH as INT) as DW_FAIXA_AGING_PROX_FECH,
            try_cast(DW_TIPO_FATURAMENTO as INT) as DW_TIPO_FATURAMENTO,
            try_cast(COD_PLATAFORMA as STRING) as COD_PLATAFORMA,
            case 
                when trim(DAT_CRIACAO_REGISTRO_TRANS) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_CRIACAO_REGISTRO_TRANS), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_CRIACAO_REGISTRO_TRANS,
            case 
                when trim(DAT_ALTERACAO_REGISTRO_TRANS) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_ALTERACAO_REGISTRO_TRANS), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_ALTERACAO_REGISTRO_TRANS,
            case 
                when trim(DAT_CANCELAMENTO_FAT) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_CANCELAMENTO_FAT), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_CANCELAMENTO_FAT,
            case 
                when trim(DAT_ORIGINAL_VCTO_FAT) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_ORIGINAL_VCTO_FAT), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_ORIGINAL_VCTO_FAT,
            case 
                when trim(DAT_ALTERACAO_VCTO_FAT) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_ALTERACAO_VCTO_FAT), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_ALTERACAO_VCTO_FAT,
            case 
                when trim(DAT_CRIACAO_FAT) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_CRIACAO_FAT), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_CRIACAO_FAT,
            case 
                when trim(DAT_VENCIMENTO_FAT) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_VENCIMENTO_FAT), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_VENCIMENTO_FAT,
            case 
                when trim(DAT_STATUS_FAT) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_STATUS_FAT), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_STATUS_FAT,
            case 
                when trim(DAT_MIN_VENCIMENTO_FAT) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_MIN_VENCIMENTO_FAT), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_MIN_VENCIMENTO_FAT,
            try_cast(NUM_BILL_SEQ_FAT as INT) as NUM_BILL_SEQ_FAT,
            try_cast(NUM_SEQ_ACORDO_FAT as INT) as NUM_SEQ_ACORDO_FAT,
            try_cast(IND_ISENCAO_COB_FAT as INT) as IND_ISENCAO_COB_FAT,
            try_cast(IND_WO as STRING) as IND_WO,
            try_cast(IND_PDD as STRING) as IND_PDD,
            try_cast(IND_PCCR as STRING) as IND_PCCR,
            try_cast(IND_ACA as STRING) as IND_ACA,
            try_cast(IND_PRIMEIRA_FAT as STRING) as IND_PRIMEIRA_FAT,
            try_cast(IND_FRAUDE as STRING) as IND_FRAUDE,
            try_cast(VAL_FAT_LIQUIDO as DECIMAL(10,2)) as VAL_FAT_LIQUIDO,
            try_cast(VAL_FAT_BRUTO as DECIMAL(10,2)) as VAL_FAT_BRUTO,
            try_cast(VAL_FAT_CREDITO as DECIMAL(10,2)) as VAL_FAT_CREDITO,
            try_cast(VAL_FAT_AJUSTE as DECIMAL(10,2)) as VAL_FAT_AJUSTE,
            try_cast(VAL_FAT_BRUTO_BC as DECIMAL(10,2)) as VAL_FAT_BRUTO_BC,
            try_cast(VAL_FAT_PAGAMENTO_BRUTO as DECIMAL(10,2)) as VAL_FAT_PAGAMENTO_BRUTO,
            try_cast(VAL_FAT_ABERTO as DECIMAL(10,2)) as VAL_FAT_ABERTO,
            try_cast(VAL_FAT_ABERTO_LIQ as DECIMAL(10,2)) as VAL_FAT_ABERTO_LIQ,
            try_cast(VAL_MULTA_JUROS as DECIMAL(10,2)) as VAL_MULTA_JUROS,
            try_cast(VAL_MULTA_CANCELAMENTO as DECIMAL(10,2)) as VAL_MULTA_CANCELAMENTO,
            try_cast(VAL_PARC_APARELHO_LIQ as DECIMAL(10,2)) as VAL_PARC_APARELHO_LIQ,
            try_cast(VAL_FAT_LIQ_JM_MC as DECIMAL(10,2)) as VAL_FAT_LIQ_JM_MC,
            case 
                when trim(DAT_ATIVACAO_CONTA_CLI) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_ATIVACAO_CONTA_CLI), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_ATIVACAO_CONTA_CLI,
            case 
                when trim(DAT_CRIACAO_DW) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_CRIACAO_DW), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_CRIACAO_DW,
            {pdthproc} as DATPROC

        from
            raw_00_com_safra
            
    """.format(pdthproc=dthproc))
lake.createOrReplaceTempView("lake")
lake.count()  

31611316

In [11]:
lake.printSchema()

root
 |-- NUM_CPF: string (nullable = true)
 |-- SAFRA: integer (nullable = true)
 |-- DAT_REFERENCIA: timestamp (nullable = true)
 |-- NUM_FATURA_HASH: string (nullable = true)
 |-- NUM_ENT_SEQ_FATURA: integer (nullable = true)
 |-- CONTRATO: long (nullable = true)
 |-- DW_UN_NEGOCIO: integer (nullable = true)
 |-- DW_HIS_PONTO_VENDA_COMTA: long (nullable = true)
 |-- DW_NUM_CLIENTE: long (nullable = true)
 |-- DW_AREA: integer (nullable = true)
 |-- DW_CICLO: integer (nullable = true)
 |-- DW_TIPO_CLIENTE_CONTA: integer (nullable = true)
 |-- DW_OFERTA: integer (nullable = true)
 |-- DW_FAIXA_AGING_FATURA: integer (nullable = true)
 |-- DW_FAIXA_AGING_DIVIDA: integer (nullable = true)
 |-- DW_FAIXA_TEMPO_BASE: integer (nullable = true)
 |-- DW_FAIXA_AGING_PROX_FECH: integer (nullable = true)
 |-- DW_TIPO_FATURAMENTO: integer (nullable = true)
 |-- COD_PLATAFORMA: string (nullable = true)
 |-- DAT_CRIACAO_REGISTRO_TRANS: timestamp (nullable = true)
 |-- DAT_ALTERACAO_REGISTRO_TRANS: t

In [12]:
lake.show(5)

+-----------+------+-------------------+--------------------+------------------+---------+-------------+------------------------+--------------+-------+--------+---------------------+---------+---------------------+---------------------+-------------------+------------------------+-------------------+--------------+--------------------------+----------------------------+--------------------+---------------------+----------------------+-------------------+-------------------+-------------------+----------------------+----------------+------------------+-------------------+------+-------+--------+-------+----------------+----------+---------------+-------------+---------------+--------------+----------------+-----------------------+--------------+------------------+---------------+----------------------+---------------------+-----------------+----------------------+-------------------+--------------+
|    NUM_CPF| SAFRA|     DAT_REFERENCIA|     NUM_FATURA_HASH|NUM_ENT_SEQ_FATURA| CONTRAT

In [13]:
# Deduplicação caso aconteça de reprocessar mesma base
lake_dedup = spark.sql("""
    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY NUM_CPF, DAT_CRIACAO_DW, CONTRATO, NUM_ENT_SEQ_FATURA
                ORDER BY DATPROC DESC
            ) AS rn
        FROM lake
    ) t
    WHERE rn = 1
""")
lake_dedup.createOrReplaceTempView("lake_dedup")
lake_dedup = spark.sql("SELECT * FROM lake_dedup")
lake_dedup.count()  

31611316

In [14]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos,
        COUNT(DISTINCT CONTRATO) as contrato_distintos,
        COUNT(DISTINCT DW_NUM_CLIENTE) as num_tel_distintos
    FROM lake_dedup
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

contagem_safras.show(5, truncate=False)

+------+------------+-------------+------------------+-----------------+
|SAFRA |total_linhas|cpf_distintos|contrato_distintos|num_tel_distintos|
+------+------------+-------------+------------------+-----------------+
|202311|1173302     |572247       |655007            |655204           |
|202312|1152922     |569431       |651600            |651739           |
|202401|2278589     |634741       |731515            |733220           |
|202402|1055098     |554201       |627401            |627387           |
|202403|1074593     |558201       |631715            |631699           |
+------+------------+-------------+------------------+-----------------+
only showing top 5 rows



In [15]:
from delta.tables import DeltaTable

silver_path = "s3a://silver/base_atraso/"

# Se a tabela ainda não existir, cria do zero
if not DeltaTable.isDeltaTable(spark, silver_path):
    print("Tabela silver não existe. Criando...")

    (
        lake_dedup
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("SAFRA")
        .save(silver_path)
    )

else:
    print("Tabela silver existe. Fazendo MERGE incremental...")

    delta_silver = DeltaTable.forPath(spark, silver_path)

    (
        delta_silver.alias("t")
        .merge(
            lake_dedup.alias("s"),
            """
            t.NUM_CPF = s.NUM_CPF
            AND t.DAT_CRIACAO_DW = s.DAT_CRIACAO_DW
            AND t.CONTRATO = s.CONTRATO
            AND t.NUM_ENT_SEQ_FATURA = s.NUM_ENT_SEQ_FATURA
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


Tabela silver não existe. Criando...


In [16]:
spark.stop()